In [51]:
# ===== ROLL A: Andmete laadimine =====

import os
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

def get_data(table_name):
    data = []
    page_size = 1000
    page = 0

    while True:
        response = (
            supabase
            .table(table_name)
            .select("*")
            .range(page * page_size, (page + 1) * page_size - 1)
            .execute()
        )

        data.extend(response.data)

        if len(response.data) < page_size:
            break

        page += 1

    return pd.DataFrame(data)

# Sales tabel
df_sales = get_data("sales")
print("df_sales shape:", df_sales.shape)
print(df_sales.dtypes)
display(df_sales.head())

# Customers tabel
df_customers = get_data("customers")
print("df_customers shape:", df_customers.shape)
print(df_customers.dtypes)
display(df_customers.head())

# Merge
df = pd.merge(df_sales, df_customers, on="customer_id", how="left")

print("Liidetud df shape:", df.shape)
print(df.dtypes)
display(df.head())

df_sales shape: (10118, 12)
id                  int64
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
dtype: object


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


df_customers shape: (3150, 9)
customer_id          int64
first_name             str
last_name              str
email                  str
phone                  str
city                   str
registration_date      str
loyalty_tier           str
birth_year           int64
dtype: object


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2637,Reet,Nurk,NaN,+372 5354 8280,Tallinn,2022-12-09,gold,1976
1,2723,Reet,Kuusik,NaN,+372 5603 8700,Tartu,2023-04-09,NaN,1998
2,2784,Mart,Pihl,NaN,+372 5536 5657,Tallinn,2022-10-30,gold,1989
3,3404,Maie,Tammik,NaN,+372 5291 9215,Tallinn,2020-03-26,NaN,2000
4,4278,Nele,Orav,NaN,+372 8700 9137,Tallinn,2024-05-10,NaN,1992


Liidetud df shape: (10118, 20)
id                     int64
sale_id                int64
invoice_id               str
sale_date                str
customer_id          float64
product_id             int64
quantity               int64
unit_price           float64
total_price          float64
channel                  str
store_location           str
payment_method           str
first_name               str
last_name                str
email                    str
phone                    str
city                     str
registration_date        str
loyalty_tier             str
birth_year           float64
dtype: object


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0


In [52]:
# ROLL B: Andmete puhastamine

# 1. Esialgne shape
print("Esialgne shape:", df.shape)

# 2. Duplikaadid
print("\nDuplikaadid:", df.duplicated().sum())
df = df.drop_duplicates()
print("Pärast duplikaatide eemaldamist:", df.shape)

# 3. NULL väärtused
print("\nNULL-id:\n", df.isnull().sum())
df = df.dropna(subset=['customer_id', 'sale_date', 'total_price'])
print("Pärast NULL eemaldamist:", df.shape)

# 4. Kuupäevade parsimine
df["sale_date"] = pd.to_datetime(
    df["sale_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

df = df.dropna(subset=["sale_date"])

# Kasutame ainult usaldusväärseid müügiandmeid kuni 31.12.2024
df = df[df["sale_date"] <= pd.to_datetime("2024-12-31")]

print("sale_date tüüp:", df["sale_date"].dtype)

# 5. Outlier'id - negatiivsed hinnad
print("\nNegatiivsed total_price väärtused:", (df['total_price'] <= 0).sum())
df = df[df['total_price'] > 0]

# 6. Puhastusraport
print("\n--- PUHASTUSRAPORT ---")
print(f"Lõplik shape: {df.shape}")
print(f"Unikaalseid kliente: {df['customer_id'].nunique()}")
print(f"Kuupäevavahemik: {df['sale_date'].min()} kuni {df['sale_date'].max()}")

Esialgne shape: (10118, 20)

Duplikaadid: 0
Pärast duplikaatide eemaldamist: (10118, 20)

NULL-id:
 id                      0
sale_id                 0
invoice_id              0
sale_date               0
customer_id           988
product_id              0
quantity                0
unit_price              0
total_price             0
channel                 0
store_location       3462
payment_method          0
first_name            988
last_name             988
email                1944
phone                 988
city                  988
registration_date     988
loyalty_tier         4660
birth_year            988
dtype: int64
Pärast NULL eemaldamist: (9130, 20)
sale_date tüüp: datetime64[us]

Negatiivsed total_price väärtused: 170

--- PUHASTUSRAPORT ---
Lõplik shape: (8325, 20)
Unikaalseid kliente: 2454
Kuupäevavahemik: 2023-01-01 00:00:00 kuni 2024-12-31 00:00:00


In [53]:
print("Negatiivseid total_price väärtusi:", (df['total_price'] <= 0).sum())
df = df[df['total_price'] > 0]
print("Pärast outlier eemaldamist:", df.shape)

Negatiivseid total_price väärtusi: 0
Pärast outlier eemaldamist: (8325, 20)


In [54]:
# ===== ROLL C: RFM segmenteerimine =====

# 1. Viitekuupäev
today = df["sale_date"].max() + pd.Timedelta(days=1)

# 2. Recency — mitu päeva on möödunud viimasest ostust
recency = df.groupby('customer_id')['sale_date'].max().reset_index()
recency.columns = ['customer_id', 'last_purchase']
recency['recency_days'] = (today - recency['last_purchase']).dt.days

# 3. Frequency — ostude arv
frequency = df.groupby('customer_id')['sale_id'].count().reset_index()
frequency.columns = ['customer_id', 'frequency']

# 4. Monetary — kogukulutus
monetary = df.groupby('customer_id')['total_price'].sum().reset_index()
monetary.columns = ['customer_id', 'monetary']

# 5. Ühenda RFM tabel
rfm = recency.merge(frequency, on='customer_id').merge(monetary, on='customer_id')

# 6. Skoorid 1-5
rfm['R_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1,2,3,4,5])

rfm['RFM_Score'] = rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)

# 7. Segmendid
def segment(score):
    if score >= 13: return 'VIP Champions'
    elif score >= 10: return 'Loyal'
    elif score >= 7:  return 'Potential'
    elif score >= 4:  return 'At Risk'
    else:             return 'Lost'

rfm['segment'] = rfm['RFM_Score'].apply(segment)

# 8. Kokkuvõte
summary = rfm['segment'].value_counts().reset_index()
summary.columns = ['segment', 'arv']
summary['osakaal_%'] = (summary['arv'] / len(rfm) * 100).round(1)
print(summary)

         segment  arv  osakaal_%
0      Potential  713       29.1
1          Loyal  697       28.4
2        At Risk  502       20.5
3  VIP Champions  422       17.2
4           Lost  120        4.9


In [64]:
# ===== ROLL D: Visualiseerimine ja leiud =====

import pandas as pd
import plotly.express as px

# --- DACA / UrbanStyle värvipalett ---
TEAL = "#009B8D"       # peamine aktsent
NAVY = "#1A1A2E"       # tekst ja teljed
GRAY = "#6B7280"       # tugiinfo
LIGHT_GRAY = "#F3F4F6" # taust
WHITE = "#FFFFFF"      # graafiku taust
BLUE = "#0072B2"       # positiivne / lojaalne
AMBER = "#E69F00"      # hoiatus / potentsiaal
ORANGE = "#D55E00"     # risk

segment_order = [
    "VIP Champions",
    "Loyal",
    "Potential",
    "At Risk",
    "Lost"
]

segment_colors = {
    "VIP Champions": TEAL,
    "Loyal": BLUE,
    "Potential": AMBER,
    "At Risk": ORANGE,
    "Lost": GRAY
}

CHART_WIDTH = 760
CHART_HEIGHT = 430

BASE_LAYOUT = dict(
    plot_bgcolor=WHITE,
    paper_bgcolor=WHITE,
    font=dict(color=NAVY, size=11),
    title=dict(font=dict(color=NAVY, size=17), x=0.02),
)

# --- Lisa RFM tabelile kliendi nimi ---
customer_cols = ["customer_id"]

if "first_name" in df.columns:
    customer_cols.append("first_name")

if "last_name" in df.columns:
    customer_cols.append("last_name")

customer_info = (
    df[customer_cols]
    .drop_duplicates(subset=["customer_id"])
)

rfm_viz = rfm.merge(customer_info, on="customer_id", how="left")

# Kliendi nime loomine: kui nimi puudub, näitame korrektset ID-d
if "first_name" in rfm_viz.columns and "last_name" in rfm_viz.columns:
    rfm_viz["customer_name"] = (
        rfm_viz["first_name"].fillna("") + " " + rfm_viz["last_name"].fillna("")
    ).str.strip()
elif "first_name" in rfm_viz.columns:
    rfm_viz["customer_name"] = rfm_viz["first_name"].fillna("")
else:
    rfm_viz["customer_name"] = ""

rfm_viz["customer_id_clean"] = (
    pd.to_numeric(rfm_viz["customer_id"], errors="coerce")
    .astype("Int64")
    .astype(str)
)

rfm_viz["customer_name"] = rfm_viz["customer_name"].replace("", pd.NA)
rfm_viz["customer_name"] = rfm_viz["customer_name"].fillna(
    "ID " + rfm_viz["customer_id_clean"]
)

rfm_viz["segment"] = pd.Categorical(
    rfm_viz["segment"],
    categories=segment_order,
    ordered=True
)

In [65]:
segment_counts = (
    rfm_viz["segment"]
    .value_counts()
    .reset_index()
)

segment_counts.columns = ["segment", "klientide_arv"]
segment_counts = segment_counts.sort_values("klientide_arv", ascending=False)

fig1 = px.bar(
    segment_counts,
    x="segment",
    y="klientide_arv",
    color="segment",
    color_discrete_map=segment_colors,
    text="klientide_arv",
    title="UrbanStyle kliendisegmentide jaotus",
    labels={
        "segment": "Kliendisegment",
        "klientide_arv": "Klientide arv"
    },
    category_orders={
        "segment": segment_counts["segment"].tolist()
    },
    width=CHART_WIDTH,
    height=CHART_HEIGHT
)

fig1.update_traces(
    textposition="outside",
    marker_line_width=0,
    textfont=dict(size=11, color=NAVY)
)

fig1.update_layout(
    **BASE_LAYOUT,
    showlegend=False,
    xaxis=dict(
        title="",
        tickfont=dict(size=10),
        tickangle=-10,
        showgrid=False
    ),
    yaxis=dict(
        title="Klientide arv",
        title_font=dict(size=11),
        tickfont=dict(size=10),
        gridcolor=LIGHT_GRAY
    ),
    margin=dict(l=70, r=30, t=70, b=75)
)

fig1.show()

In [66]:
segment_profile = (
    rfm_viz
    .groupby("segment", observed=True)
    .agg(
        avg_recency=("recency_days", "mean"),
        avg_monetary=("monetary", "mean"),
        customers=("customer_id", "count")
    )
    .reset_index()
)

fig2 = px.scatter(
    segment_profile,
    x="avg_recency",
    y="avg_monetary",
    size="customers",
    color="segment",
    text="segment",
    color_discrete_map=segment_colors,
    category_orders={"segment": segment_order},
    title="RFM segmentide profiil",
    labels={
        "avg_recency": "Päevi viimasest ostust",
        "avg_monetary": "Keskm. kogukulutus",
        "customers": "Klientide arv",
        "segment": "Segment"
    },
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
    size_max=44
)

fig2.update_traces(
    textposition="top center",
    marker=dict(
        opacity=0.78,
        line=dict(width=1, color=WHITE)
    ),
    textfont=dict(size=10, color=NAVY)
)

fig2.update_layout(
    **BASE_LAYOUT,
    showlegend=False,
    xaxis=dict(
        title="Päevi viimasest ostust",
        title_font=dict(size=11),
        tickfont=dict(size=10),
        gridcolor=LIGHT_GRAY
    ),
    yaxis=dict(
        title="Keskm. kogukulutus",
        title_font=dict(size=11),
        tickfont=dict(size=10),
        gridcolor=LIGHT_GRAY,
        tickformat=",.0f"
    ),
    margin=dict(l=75, r=35, t=70, b=70)
)

fig2.show()

In [67]:
top_vip = (
    rfm_viz[rfm_viz["segment"] == "VIP Champions"]
    .nlargest(10, "monetary")
    .sort_values("monetary", ascending=False)
    .copy()
)

top_vip["monetary_label"] = top_vip["monetary"].map(lambda x: f"{x/1000:.1f}k")

vip_order = top_vip["customer_name"].tolist()

fig3 = px.bar(
    top_vip,
    x="monetary",
    y="customer_name",
    orientation="h",
    text="monetary_label",
    color_discrete_sequence=[TEAL],
    title="Top 10 VIP klienti",
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
    labels={
        "customer_name": "",
        "monetary": "Kogukulutus"
    }
)

fig3.update_traces(
    textposition="inside",
    insidetextanchor="end",
    textfont=dict(color=WHITE, size=11),
    marker_line_width=0,
    hovertemplate="<b>%{y}</b><br>Kogukulutus: %{x:,.2f} EUR<extra></extra>"
)

fig3.update_layout(
    **BASE_LAYOUT,
    showlegend=False,
    xaxis=dict(
        title="Kogukulutus",
        title_font=dict(size=11),
        tickfont=dict(size=10),
        gridcolor=LIGHT_GRAY,
        tickformat=",.0f"
    ),
    yaxis=dict(
        title="",
        tickfont=dict(size=10),
        showgrid=False,
        categoryorder="array",
        categoryarray=vip_order[::-1]
    ),
    margin=dict(l=120, r=35, t=70, b=70)
)

fig3.show()

In [63]:
# ===== Äritõlgendus Markole =====

total_revenue = rfm_viz["monetary"].sum()

vip_customers = rfm_viz[rfm_viz["segment"] == "VIP Champions"]
at_risk_customers = rfm_viz[rfm_viz["segment"] == "At Risk"]
lost_customers = rfm_viz[rfm_viz["segment"] == "Lost"]
potential_customers = rfm_viz[rfm_viz["segment"] == "Potential"]

vip_count = len(vip_customers)
vip_revenue = vip_customers["monetary"].sum()
vip_revenue_share = vip_revenue / total_revenue * 100

at_risk_count = len(at_risk_customers)
lost_count = len(lost_customers)
potential_count = len(potential_customers)

print(f"""
Äritõlgendus Markole:
UrbanStyle'il on {vip_count} VIP Champions segmenti kuuluvat klienti, kes annavad kokku {vip_revenue_share:.2f}% kogukäibest.
Need kliendid on kõige väärtuslikum segment, sest nad ostavad sageli ja kulutavad palju.
At Risk segmenti kuulub {at_risk_count} klienti ning Lost segmenti {lost_count} klienti, kelle puhul on oluline vähendada kliendikadu.
Potential segmenti kuulub {potential_count} klienti, keda saab sihitud pakkumistega kasvatada lojaalsemateks klientideks.

Soovitused Markole:

VIP programm — paku {vip_count} VIP Champions kliendile eritingimused: varajane juurdepääs 
uutele kollektsioonidele, tasuta saatmine ja personaalsem klienditeenindus. Need kliendid 
annavad suure osa käibest ning nende hoidmine on odavam kui uute klientide leidmine.

Win-back kampaania — saada {at_risk_count} At Risk kliendile personaalne pakkumine järgmise 
30 päeva jooksul, näiteks 15% allahindlus või eripakkumine nende varasema ostukäitumise põhjal. 
Kui nad ei reageeri, võivad nad liikuda Lost segmenti.

Nurture programm — {potential_count} Potential klienti on kasvupotentsiaaliga. 
Lojaalsusprogramm, sihitud pakkumised ja regulaarsed soovitused aitavad neid kasvatada 
Loyal või VIP tasemele.

Lojaalsustaseme kontroll — võrdle RFM segmente ettevõtte olemasoleva loyalty_tier väljaga. 
Kui klient kuulub RFM järgi VIP Champions või Loyal segmenti, aga tema loyalty_tier puudub 
või on madal, tasub talle pakkuda lojaalsusprogrammiga liitumist või kõrgemat taset. 
Nii saab lojaalsusprogramm paremini peegeldada tegelikku ostukäitumist.
""")


Äritõlgendus Markole:
UrbanStyle'il on 422 VIP Champions segmenti kuuluvat klienti, kes annavad kokku 41.67% kogukäibest.
Need kliendid on kõige väärtuslikum segment, sest nad ostavad sageli ja kulutavad palju.
At Risk segmenti kuulub 502 klienti ning Lost segmenti 120 klienti, kelle puhul on oluline vähendada kliendikadu.
Potential segmenti kuulub 713 klienti, keda saab sihitud pakkumistega kasvatada lojaalsemateks klientideks.

Soovitused Markole:

VIP programm — paku 422 VIP Champions kliendile eritingimused: varajane juurdepääs 
uutele kollektsioonidele, tasuta saatmine ja personaalsem klienditeenindus. Need kliendid 
annavad suure osa käibest ning nende hoidmine on odavam kui uute klientide leidmine.

Win-back kampaania — saada 502 At Risk kliendile personaalne pakkumine järgmise 
30 päeva jooksul, näiteks 15% allahindlus või eripakkumine nende varasema ostukäitumise põhjal. 
Kui nad ei reageeri, võivad nad liikuda Lost segmenti.

Nurture programm — 713 Potential klienti on kasvu